# Task: GNN training on synthetic SPV simulations: adjacency, cell state and property matrices

We will be developing a graph neural network (GNN)-based model capable of inferring mechanistic rules and uncovering the principles driving DPAC aggregation. To facilitate this, the GNN will initially be trained using synthetic Self-Propelled Voronoi (SPV) simulations, serving as placeholder data while the deep learning infrastructure is optimized. The GNN will be validated by its ability to, first, recover the physical mechanisms embedded in the SPV model, then subsequently applied to DPAC data to explore the impacts of initial thickness and cell density.

### GNN training

In [25]:
import os
import numpy as np
import networkx as nx
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import DataLoader
from torch_geometric.data import Data
from torch_geometric.utils import from_networkx
from torch_geometric.nn import GINEConv, global_mean_pool

from torch.optim import AdamW
from sklearn.model_selection import KFold

##########################################################################
# 1. Parameter Parser
##########################################################################

def parse_parameters(param_path):
    """
    Reads e.g. W, v0, Dr, ...
    """
    params = {}
    print(f"Loading param file={param_path}")
    with open(param_path, "r") as f:
        exec(f.read(), {}, params)
    if not isinstance(params["W"], np.ndarray):
        params["W"] = np.array(params["W"])
    return params

##########################################################################
# 2. Data Loading with Caching
##########################################################################

def load_matrices(directory, timepoint):
    """
    Loads (graph_mat, properties_mat, state_mat) from .npy files.
    Uses an internal cache dict for speed.
    """
    if not hasattr(load_matrices, 'cache'):
        load_matrices.cache={}
    cache_key=(directory,timepoint)
    if cache_key not in load_matrices.cache:
        print(f"Load timepoint={timepoint} from dir={directory}")
        graph_mat= np.load(os.path.join(directory,f"{timepoint}_graph_mat.npy"))
        props_mat= np.load(os.path.join(directory,f"{timepoint}_properties_mat.npy"))
        state_mat= np.load(os.path.join(directory,f"{timepoint}_state_mat.npy"))
        load_matrices.cache[cache_key]=(graph_mat,props_mat,state_mat)
    return load_matrices.cache[cache_key]

##########################################################################
# 3. GCA init
##########################################################################

def initialize_gca(graph_mat, properties_mat, state_mat, params):
    """
    Builds a Graph with node+edge attributes; node states remain constant.
    """
    g= nx.from_numpy_array(graph_mat, create_using=nx.Graph)
    for i,(area,perim) in enumerate(properties_mat):
        ctype= np.argmax(state_mat[i])
        g.nodes[i]["state"]= state_mat[i]
        g.nodes[i].update({
            "area":max(area,0),
            "perimeter":max(perim,0),
            "motility":params["v0"][ctype],
            "persistence":params["Dr"],
            "kappa_A":params["kappa_A"],
            "kappa_P":params["kappa_P"],
            "A0":params["A0"][ctype],
            "P0":params["P0"][ctype],
        })
    for (u,v) in g.edges():
        t_u= np.argmax(g.nodes[u]["state"])
        t_v= np.argmax(g.nodes[v]["state"])
        adh= params["W"][t_u][t_v]
        g.edges[u,v].update({
            "adhesion": adh,
            "repulsion_radius": params["a"],
            "repulsion_coefficient": params["k"]
        })
    g.graph["adj_matrix"]= graph_mat
    return g

##########################################################################
# 4. Build single (t->t+interval) pair as PyG Data
##########################################################################

def build_pyg_from_gca(g_current, g_next, num_cell_types=2):
    """
    Creates PyG 'Data':
      node features => (area, perimeter, state, rel_diff)
      edge features => (adh, radius, coeff, edge_type_flag)
      next_area, next_perim, next_adj, next_state, ...
    """
    data= from_networkx(g_current)
    n= g_current.number_of_nodes()

    # partial feats => area, perimeter, state
    node_feat=[]
    type1_count=0
    for i in range(n):
        a= g_current.nodes[i]["area"]
        p= g_current.nodes[i]["perimeter"]
        st= g_current.nodes[i]["state"]
        node_feat.append(np.concatenate([[a,p], st]))
    node_feat= np.array(node_feat,dtype=np.float32)

    # global fraction type1
    for i in range(n):
        if np.argmax(node_feat[i,2:2+num_cell_types])==1:
            type1_count+=1
    global_t1_frac= float(type1_count)/float(n) if n>0 else 0.

    # local fraction
    local_fracs= np.zeros(n,dtype=np.float32)
    for i in range(n):
        nbrs= list(g_current.neighbors(i))
        if len(nbrs)==0:
            local_fracs[i]=0.
            continue
        c=0
        for nbr in nbrs:
            if np.argmax(node_feat[nbr,2:2+num_cell_types])==1:
                c+=1
        local_fracs[i]= c/ float(len(nbrs))

    rel_diff= local_fracs- global_t1_frac
    # final node features => shape= (2+num_cell_types+1)
    node_feat= np.hstack([node_feat, rel_diff.reshape(-1,1)])
    data.x= torch.tensor(node_feat,dtype=torch.float)

    # edge feats => 4 dims: (adh, radius, coeff, edge_type_flag)
    edge_feats=[]
    row,col= data.edge_index
    for idx in range(row.shape[0]):
        u= row[idx].item()
        v= col[idx].item()
        e= g_current.edges[u,v]
        feats_edge= [
            e["adhesion"],
            e["repulsion_radius"],
            e["repulsion_coefficient"]
        ]
        # edge_type_flag => 1 if both type=1
        node_type_u= np.argmax(node_feat[u, 2:2+num_cell_types])
        node_type_v= np.argmax(node_feat[v, 2:2+num_cell_types])
        edge_flag= 1 if (node_type_u==1 and node_type_v==1) else 0
        feats_edge.append(edge_flag)
        edge_feats.append(feats_edge)
    data.edge_attr= torch.tensor(edge_feats,dtype=torch.float)

    # next adjacency
    mat_nxt= nx.to_numpy_array(g_next)
    data.next_adj= torch.tensor(mat_nxt[row,col],dtype=torch.float)
    # next area/perim
    data.next_area= torch.tensor([g_next.nodes[i]["area"] for i in range(n)],dtype=torch.float)
    data.next_perim= torch.tensor([g_next.nodes[i]["perimeter"] for i in range(n)],dtype=torch.float)
    # next state
    data.next_state= torch.tensor([g_next.nodes[i]["state"] for i in range(n)],dtype=torch.float)

    return data

##########################################################################
# 5. GNN model
##########################################################################

class DeepGraphPredictor(nn.Module):
    """
    4-layer GINEConv => output state_pred, area_pred, perim_pred, adj_pred
    (edge_dim=4 => (adh, radius, coeff, type1flag))
    """
    def __init__(self, node_dim, edge_dim, hidden_dim, num_cell_types):
        super().__init__()
        self.node_encoder= nn.Linear(node_dim, hidden_dim)

        self.mlp1= nn.Sequential(
            nn.Linear(hidden_dim,hidden_dim),nn.ReLU(),
            nn.Linear(hidden_dim,hidden_dim)
        )
        self.mlp2= nn.Sequential(
            nn.Linear(hidden_dim,hidden_dim),nn.ReLU(),
            nn.Linear(hidden_dim,hidden_dim)
        )
        self.mlp3= nn.Sequential(
            nn.Linear(hidden_dim,hidden_dim),nn.ReLU(),
            nn.Linear(hidden_dim,hidden_dim)
        )
        self.mlp4= nn.Sequential(
            nn.Linear(hidden_dim,hidden_dim),nn.ReLU(),
            nn.Linear(hidden_dim,hidden_dim)
        )

        self.conv1= GINEConv(self.mlp1, edge_dim=edge_dim)
        self.conv2= GINEConv(self.mlp2, edge_dim=edge_dim)
        self.conv3= GINEConv(self.mlp3, edge_dim=edge_dim)
        self.conv4= GINEConv(self.mlp4, edge_dim=edge_dim)

        self.state_decoder= nn.Linear(hidden_dim, num_cell_types)
        self.area_decoder=  nn.Linear(hidden_dim, 1)
        self.perim_decoder= nn.Linear(hidden_dim, 1)
        self.adj_decoder=   nn.Linear(2*hidden_dim,1)

    def forward(self,x,edge_index,edge_attr,batch=None):
        x= self.node_encoder(x)
        x= self.conv1(x,edge_index,edge_attr)
        x= F.relu(x)
        x= self.conv2(x,edge_index,edge_attr)
        x= F.relu(x)

        if batch is None:
            g= x.mean(dim=0,keepdim=True)
            x= x+ g
        else:
            g= global_mean_pool(x,batch)
            x= x+ g[batch]

        x= self.conv3(x,edge_index,edge_attr)
        x= F.relu(x)
        x= self.conv4(x,edge_index,edge_attr)
        x= F.relu(x)

        state_pred= F.log_softmax(self.state_decoder(x),dim=-1)
        area_pred=  self.area_decoder(x)
        perim_pred= self.perim_decoder(x)

        row,col= edge_index
        x_u= x[row]
        x_v= x[col]
        edge_repr= torch.cat([x_u,x_v],dim=1)
        adj_pred= self.adj_decoder(edge_repr)

        return state_pred, area_pred, perim_pred, adj_pred

##########################################################################
# 6. Train/Validate "One Epoch" - no shuffle in time
##########################################################################

def run_one_epoch(model, data_dirs, param_list, timepoints, timepoint_interval,
                  num_cell_types, device, optimizer=None, is_train=True):
    """
    Goes through the directories in order, each in time order: (t -> t+interval)
    If is_train=True => do backward & step.
    Else => no grad => just compute loss.
    Returns total_loss and count of pairs, so we can compute average.
    """
    if is_train:
        model.train()
    else:
        model.eval()

    total_loss= 0.0
    total_pairs= 0

    for sim_idx, (data_dir, params) in enumerate(zip(data_dirs, param_list)):
        t=0
        while t< timepoints:
            nxt= t+ timepoint_interval
            if nxt> timepoints:
                break

            gm_cur, pm_cur, sm_cur= load_matrices(data_dir,t)
            gm_nxt, pm_nxt, sm_nxt= load_matrices(data_dir,nxt)

            g_cur= initialize_gca(gm_cur, pm_cur, sm_cur, params)
            g_nxt= initialize_gca(gm_nxt, pm_nxt, sm_nxt, params)

            data_pyg= build_pyg_from_gca(g_cur, g_nxt, num_cell_types)
            data_pyg= data_pyg.to(device)

            with torch.set_grad_enabled(is_train):
                state_pred, area_pred, perim_pred, adj_pred= model(
                    data_pyg.x, data_pyg.edge_index, data_pyg.edge_attr
                )
                # adjacency
                loss_adj= F.binary_cross_entropy_with_logits(
                    adj_pred.squeeze(), data_pyg.next_adj.squeeze()
                )
                # area/perim => e.g. L1
                loss_area= F.smooth_l1_loss(area_pred.squeeze(), data_pyg.next_area)
                loss_perim=F.smooth_l1_loss(perim_pred.squeeze(), data_pyg.next_perim)
                # state => KLDiv
                loss_state= F.kl_div(state_pred, data_pyg.next_state, reduction='batchmean')

                batch_loss= loss_adj + loss_area + loss_perim + loss_state
                reg= 1e-4 * sum(p.pow(2).sum() for p in model.parameters())
                batch_loss+= reg

                if is_train:
                    optimizer.zero_grad()
                    batch_loss.backward()
                    nn.utils.clip_grad_norm_(model.parameters(),1.)
                    optimizer.step()

            total_loss+= batch_loss.item()
            total_pairs+=1

            t+= timepoint_interval

    return total_loss, total_pairs


##########################################################################
# 7. K-Fold across directories
##########################################################################

def k_fold_training(
    data_dirs, param_files,   # each of length 10
    timepoints, timepoint_interval,
    node_dim, edge_dim, hidden_dim, num_cell_types,
    epochs=5, lr=1e-4, k_folds=5
):
    """
    We'll do 5-fold CV across the 10 directories (2 for val, 8 for train each fold).
    Then pick best overall validation model and save it.
    """
    device= torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device={device}")

    from sklearn.model_selection import KFold
    kf= KFold(n_splits=k_folds, shuffle=True, random_state=42)

    dir_indices= list(range(len(data_dirs)))  # [0..9]
    best_model_state= None
    best_val= float('inf')

    # parse param files once => store them
    param_list= []
    for pf in param_files:
        p= parse_parameters(pf)
        param_list.append(p)

    for fold, (train_idx, val_idx) in enumerate(kf.split(dir_indices)):
        print(f"\n=== Fold {fold+1}/{k_folds} => train dirs={train_idx}, val dirs={val_idx} ===")

        # Build sub-lists
        train_data_dirs= [ data_dirs[i] for i in train_idx ]
        train_params=    [ param_list[i] for i in train_idx ]
        val_data_dirs=   [ data_dirs[i] for i in val_idx ]
        val_params=      [ param_list[i] for i in val_idx ]

        # fresh model
        model= DeepGraphPredictor(node_dim, edge_dim, hidden_dim, num_cell_types).to(device)
        optimizer= AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

        fold_best= float('inf')
        fold_best_state= None
        patience= 0

        for epoch in range(epochs):
            print(f"\n--- EPOCH {epoch+1}/{epochs} (Fold {fold+1}) ---")
            # train
            train_loss, train_pairs = run_one_epoch(
                model,
                train_data_dirs, train_params,
                timepoints, timepoint_interval,
                num_cell_types,
                device,
                optimizer=optimizer,
                is_train=True
            )
            avg_train= train_loss/train_pairs if train_pairs>0 else 0.
            print(f"[Fold {fold+1}][Epoch {epoch+1}] train avg={avg_train:.4f}")

            # val
            val_loss, val_pairs = run_one_epoch(
                model,
                val_data_dirs, val_params,
                timepoints, timepoint_interval,
                num_cell_types,
                device,
                optimizer=None,
                is_train=False
            )
            avg_val= val_loss/val_pairs if val_pairs>0 else 0.
            print(f"[Fold {fold+1}][Epoch {epoch+1}] val avg={avg_val:.4f}")

            # check best
            if avg_val< fold_best:
                fold_best= avg_val
                fold_best_state= model.state_dict()
                patience= 0
                print("   --> new best for this fold.")
            else:
                patience+=1
                if patience>=10:
                    print("   --> early stopping triggered.")
                    break

        # fold done => check if fold_best < best_val
        if fold_best< best_val:
            best_val= fold_best
            best_model_state= fold_best_state

    # end all folds => save best overall
    if best_model_state is not None:
        final_model= DeepGraphPredictor(node_dim, edge_dim, hidden_dim, num_cell_types)
        final_model.load_state_dict(best_model_state)
        torch.save(final_model.state_dict(), "best_model.pth")
        print(f"\nDone. Best validation loss across folds={best_val:.4f}")
        print("Saved best_model.pth.")
        return final_model
    else:
        print("No best model found??")
        return None

##########################################################################
# 8. Main Execution
##########################################################################

if __name__ == "__main__":
    data_dirs = [
        f"/Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/output_Fig5I_matrix/{i}_matrix_output"
        for i in range(1, 11)
    ]
    param_files = [
        f"/Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/DPAC1_parameters/{i}.py"
        for i in range(1, 11)
    ]

    timepoints= 2000
    timepoint_interval= 100

    num_cell_types= 2
    node_dim= 5  # e.g. area,perim + 2 states + 1 rel_diff
    edge_dim= 4  # e.g. (adh, radius, coeff, edge_type_flag)
    hidden_dim= 64
    epochs= 10
    k_folds= 5

    best_model= k_fold_training(
        data_dirs, param_files,
        timepoints, timepoint_interval,
        node_dim, edge_dim, hidden_dim, num_cell_types,
        epochs=epochs, lr=1e-4, k_folds=k_folds
    )

    print("K-Fold training complete.")


Using device=cpu
Loading param file=/Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/DPAC1_parameters/1.py
Loading param file=/Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/DPAC1_parameters/2.py
Loading param file=/Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/DPAC1_parameters/3.py
Loading param file=/Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/DPAC1_parameters/4.py
Loading param file=/Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/DPAC1_parameters/5.py
Loading param file=/Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/DPAC1_parameters/6.py
Loading param file=/Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/DPAC1_parameters/7.py
Loading param file=/Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/DPAC1_parameters/8.py
Loading param file=/Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/DPAC1_parameters/9.py
Loading param file=/Users/sop

KeyboardInterrupt: 

Loading param file=/path/to/1.py


FileNotFoundError: [Errno 2] No such file or directory: '/path/to/1.py'

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches

import torch
import torch.nn.functional as F
from torch_geometric.utils import from_networkx
from torch_geometric.data import Data

# Suppose you have already:
#   parse_parameters(param_file)
#   initialize_gca(graph_mat, properties_mat, state_mat, params)
#   class DeepGraphPredictor(nn.Module): ...
# which are the same as your training-time definitions.

##########################################################################
# 1. Build PyG data from GCA (with 4D edges)
##########################################################################

def build_pyg_data_from_gca(graph_mat, properties_mat, state_mat, params, initialize_gca_func, num_cell_types=2):
    """
    Builds a PyG Data object from your GCA initialization logic,
    matching the 4D edge logic used in training:
      Edge features = (adhesion, repulsion_radius, repulsion_coefficient, edge_type_flag).
    Node features = (area, perimeter, node_state, rel_diff).
    """
    # 1) Build a networkx Graph from GCA
    g = initialize_gca_func(graph_mat, properties_mat, state_mat, params)
    data = from_networkx(g)
    num_nodes = g.number_of_nodes()

    # 2) Global fraction type=1
    type1_count=0
    for i in g.nodes:
        st= g.nodes[i]["state"]
        if np.argmax(st)==1:
            type1_count+=1
    global_type1_fraction= float(type1_count)/float(num_nodes) if num_nodes>0 else 0.

    # 3) Local fraction type=1 among neighbors
    local_fractions= np.zeros(num_nodes,dtype=np.float32)
    for i in range(num_nodes):
        nbrs= list(g.neighbors(i))
        if len(nbrs)==0:
            local_fractions[i]=0.
            continue
        c=0
        for nbr in nbrs:
            stnbr= g.nodes[nbr]["state"]
            if np.argmax(stnbr)==1:
                c+=1
        local_fractions[i]= c/ float(len(nbrs))

    # 4) Node features => (area, perimeter, state, rel_diff)
    node_feat=[]
    for i in range(num_nodes):
        area_i  = g.nodes[i]["area"]
        perim_i = g.nodes[i]["perimeter"]
        st_i    = g.nodes[i]["state"]    # shape [num_cell_types]
        rel_diff= local_fractions[i]- global_type1_fraction
        feats= np.concatenate([[area_i, perim_i], st_i, [rel_diff]])
        node_feat.append(feats)
    node_feat= np.array(node_feat,dtype=np.float32)
    data.x= torch.tensor(node_feat,dtype=torch.float)

    # 5) Edge features => 4D:
    #   (adhesion, repulsion_radius, repulsion_coefficient, edge_type_flag=1 if both type=1)
    edge_feats=[]
    row, col= data.edge_index
    for e_idx in range(row.size(0)):
        u= row[e_idx].item()
        v= col[e_idx].item()
        edge_attr= g.edges[u,v]
        # normal features
        feats_edge = [
            edge_attr["adhesion"],
            edge_attr["repulsion_radius"],
            edge_attr["repulsion_coefficient"]
        ]
        # compute edge_type_flag
        node_type_u= np.argmax(node_feat[u, 2:2+num_cell_types])
        node_type_v= np.argmax(node_feat[v, 2:2+num_cell_types])
        edge_type_flag= 1 if (node_type_u==1 and node_type_v==1) else 0
        feats_edge.append(edge_type_flag)  # 4th dimension
        edge_feats.append(feats_edge)

    data.edge_attr= torch.tensor(edge_feats,dtype=torch.float)

    return data


##########################################################################
# 2. Visualization / Plot Helpers
##########################################################################

def color_code_adjacency_matrix(graph_mat, node_types):
    """
    0= No edge
    1= Edge same-type=0
    2= Edge same-type=1
    3= Edge cross-type
    """
    color_coded= np.zeros_like(graph_mat,dtype=int)
    n= graph_mat.shape[0]
    for i in range(n):
        for j in range(n):
            if graph_mat[i,j]==1:
                if node_types[i]== node_types[j]:
                    if node_types[i]==0:
                        color_coded[i,j]=1
                    else:
                        color_coded[i,j]=2
                else:
                    color_coded[i,j]=3
    return color_coded

def plot_adjacency_matrix(graph_mat, state_mat, save_path):
    # if state_mat=[N,2], do argmax => node_types
    if len(state_mat.shape)==2 and state_mat.shape[1]>1:
        node_types= np.argmax(state_mat,axis=1)
    else:
        node_types= state_mat.flatten().astype(int)

    color_coded= color_code_adjacency_matrix(graph_mat,node_types)
    cmap= ListedColormap(["white","#FFA500","#800080","#0000FF"])
    plt.figure(figsize=(8,8))
    plt.title("Color-Coded Adjacency Matrix")
    plt.imshow(color_coded,cmap=cmap,interpolation="nearest")
    cbar= plt.colorbar(ticks=[0,1,2,3])
    cbar.ax.set_yticklabels([
        "No Edge",
        "Same Type=0 (Orange)",
        "Same Type=1 (Purple)",
        "Diff Type (Blue)"
    ])
    plt.xlabel("Node Index")
    plt.ylabel("Node Index")
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

def plot_graph_representation(graph_mat, state_mat, save_path, pos=None):
    g= nx.from_numpy_array(graph_mat, create_using=nx.Graph)
    if len(state_mat.shape)==2 and state_mat.shape[1]>1:
        node_types= np.argmax(state_mat,axis=1)
    else:
        node_types= state_mat.flatten().astype(int)

    node_colors=[]
    node_alphas=[]
    for t in node_types:
        if t==0:
            node_colors.append("yellow")
            node_alphas.append(0.5)
        else:
            node_colors.append("purple")
            node_alphas.append(1.0)

    if pos is None:
        pos= nx.kamada_kawai_layout(g)

    fig,ax= plt.subplots(figsize=(8,8))
    nx.draw_networkx_edges(g, pos, edge_color="gray", ax=ax)
    nx.draw_networkx_nodes(
        g, pos,
        node_color=node_colors, alpha=node_alphas, node_size=50, ax=ax
    )
    patches= [
        mpatches.Patch(color="yellow", label="Cell Type 0 (alpha=0.5)"),
        mpatches.Patch(color="purple", label="Cell Type 1 (alpha=1.0)")
    ]
    ax.legend(handles=patches, loc="upper right", title="Cell Types", fontsize="small")
    plt.title("Predicted Graph Representation")
    plt.savefig(save_path)
    plt.close()
    return pos

def plot_area_perimeter_bar(properties_mat, state_mat, save_path):
    if len(state_mat.shape)==2 and state_mat.shape[1]>1:
        node_types= np.argmax(state_mat,axis=1)
    else:
        node_types= state_mat.flatten().astype(int)

    node_colors=[]
    for t in node_types:
        if t==0:
            node_colors.append("yellow")
        else:
            node_colors.append("purple")

    fig,axes= plt.subplots(1,2,figsize=(10,4))
    # area
    axes[0].bar(range(len(properties_mat)), properties_mat[:,0], color=node_colors)
    axes[0].set_title("Predicted Cell Area")
    axes[0].set_xlabel("Node Index")
    axes[0].set_ylabel("Area")

    # perimeter
    axes[1].bar(range(len(properties_mat)), properties_mat[:,1], color=node_colors)
    axes[1].set_title("Predicted Cell Perimeter")
    axes[1].set_xlabel("Node Index")
    axes[1].set_ylabel("Perimeter")

    patches= [
        mpatches.Patch(color="yellow",label="Cell Type 0"),
        mpatches.Patch(color="purple",label="Cell Type 1")
    ]
    fig.legend(handles=patches, loc="center right", title="Cell Types", fontsize="small")
    plt.tight_layout(rect=[0,0,0.85,1])
    plt.savefig(save_path)
    plt.close()

def visualize_prediction(output_dir, timepoint, graph_mat, state_mat, properties_mat):
    adjacency_dir= os.path.join(output_dir,"adjacency_matrix")
    graph_dir= os.path.join(output_dir,"graph_representation")
    state_dir= os.path.join(output_dir,"states")
    os.makedirs(adjacency_dir,exist_ok=True)
    os.makedirs(graph_dir,exist_ok=True)
    os.makedirs(state_dir,exist_ok=True)

    adj_save_path= os.path.join(adjacency_dir, f"{timepoint}.png")
    plot_adjacency_matrix(graph_mat, state_mat, adj_save_path)

    graph_save_path= os.path.join(graph_dir, f"{timepoint}.png")
    plot_graph_representation(graph_mat, state_mat, graph_save_path)

    state_save_path= os.path.join(state_dir, f"{timepoint}.png")
    plot_area_perimeter_bar(properties_mat, state_mat, state_save_path)

    print(f"  -> Visualization saved for timepoint {timepoint}.")

##########################################################################
# 3. MAIN: Sequential Prediction + Visualization
##########################################################################

def sequential_prediction_with_visuals(
    data_dir,
    param_file,
    model_path,
    start_timepoint,
    timepoints,              # e.g. [0,100,200,...,2000]
    node_dim, edge_dim, hidden_dim, num_cell_types,
    initialize_gca_func,     # same as training
    parse_params_func,       # same as training
    DeepGraphPredictorClass, # same as training
    output_dir
):
    """
    We do consecutive predictions in steps => (timepoints[i-1] -> timepoints[i])
    building a 4D edge feature if 'edge_dim=4', i.e. adding 'edge_type_flag'.
    """
    device= torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    # 1) load model
    print(f"Loading model from {model_path}")
    model= DeepGraphPredictorClass(node_dim, edge_dim, hidden_dim, num_cell_types)
    model.load_state_dict(torch.load(model_path,map_location=device))
    model.to(device)
    model.eval()

    # 2) parse param + load initial .npy
    params= parse_params_func(param_file)

    graph_mat= np.load(os.path.join(data_dir,f"{start_timepoint}_graph_mat.npy"))
    state_mat= np.load(os.path.join(data_dir,f"{start_timepoint}_state_mat.npy"))
    properties_mat= np.load(os.path.join(data_dir,f"{start_timepoint}_properties_mat.npy"))

    os.makedirs(output_dir,exist_ok=True)

    # save baseline
    np.save(os.path.join(output_dir,f"{start_timepoint}_graph_mat.npy"), graph_mat)
    np.save(os.path.join(output_dir,f"{start_timepoint}_state_mat.npy"), state_mat)
    np.save(os.path.join(output_dir,f"{start_timepoint}_properties_mat.npy"), properties_mat)
    print(f"Initial time={start_timepoint} => saved to {output_dir}")
    visualize_prediction(output_dir, start_timepoint, graph_mat, state_mat, properties_mat)

    # 3) iterate over timepoints
    #    e.g. timepoints= [0,100,200,...,2000]
    for i in range(1, len(timepoints)):
        prev_t= timepoints[i-1]
        curr_t= timepoints[i]
        print(f"\n--- Predicting from time {prev_t} to {curr_t} ---")

        # build PyG from *previous* step
        data_pyg= build_pyg_data_from_gca(
            graph_mat,
            properties_mat,
            state_mat,
            params,
            initialize_gca_func,
            num_cell_types=num_cell_types
        )
        data_pyg= data_pyg.to(device)

        # forward pass
        with torch.no_grad():
            state_pred, area_pred, perim_pred, adj_pred= model(
                data_pyg.x, data_pyg.edge_index, data_pyg.edge_attr
            )

        # convert predictions
        # if training used log_softmax => state_pred => exp => distribution
        state_pred= state_pred.exp().cpu().numpy()
        area_pred= area_pred.cpu().numpy().reshape(-1)
        perim_pred= perim_pred.cpu().numpy().reshape(-1)
        adj_logits= adj_pred.cpu().numpy().reshape(-1)
        adj_probs= 1.0/(1.0+ np.exp(-adj_logits))

        row= data_pyg.edge_index[0].cpu().numpy()
        col= data_pyg.edge_index[1].cpu().numpy()
        num_nodes= state_pred.shape[0]
        new_graph_mat= np.zeros((num_nodes,num_nodes),dtype=np.float32)

        # threshold for adjacency!!!
        threshold=0.15
        for e_idx in range(len(adj_probs)):
            if adj_probs[e_idx]>= threshold:
                u= row[e_idx]
                v= col[e_idx]
                new_graph_mat[u,v]=1.
                new_graph_mat[v,u]=1.

        # node states remain constant => new_state_mat= state_mat
        # or if you want the predicted distribution => new_state_mat= state_pred
        new_state_mat= state_mat
        # area/perim => from the model
        new_properties_mat= np.stack([area_pred, perim_pred], axis=1)

        # save
        np.save(os.path.join(output_dir, f"{curr_t}_graph_mat.npy"), new_graph_mat)
        np.save(os.path.join(output_dir, f"{curr_t}_state_mat.npy"), new_state_mat)
        np.save(os.path.join(output_dir, f"{curr_t}_properties_mat.npy"), new_properties_mat)

        visualize_prediction(output_dir, curr_t, new_graph_mat, new_state_mat, new_properties_mat)

        # update references
        graph_mat= new_graph_mat
        state_mat= new_state_mat
        properties_mat= new_properties_mat

    print("\nSequential prediction + visualization complete.")

# ---------------------------------------------------
# EXAMPLE STANDALONE USAGE
# ---------------------------------------------------
if __name__ == "__main__":
    # Suppose you have the following from your training phase:
    #
    # from train_script import (
    #    parse_parameters,       # param_file -> dict of {W, v0, Dr, kappa_A, etc.}
    #    initialize_gca,        # graph_mat, properties_mat, state_mat, params -> Nx Graph
    #    GraphPredictor         # your GINEConv-based model class
    # )
    #
    # We'll just assume they're available in the namespace.

    data_dir = "/Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/matrix_output_1_test"
    param_file = "/Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/DPAC1_parameters/1.py"
    output_dir = "/Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/predictions"

    model_path = "best_model_across_5folds.pth"  # trained model

    start_timepoint= 0
    # step by 100 => 0,100,200,...,2000
    timepoints= list(range(0,2001,100))

    num_cell_types= 2
    node_dim= 5   # e.g. area,perim + 2 states + rel_diff
    edge_dim= 4   # or 4 if you used edge_type_flag in training
    hidden_dim= 64

    sequential_prediction_with_visuals(
        data_dir,
        param_file,
        model_path,
        start_timepoint,
        timepoints,
        node_dim, edge_dim, hidden_dim, num_cell_types,
        initialize_gca_func= initialize_gca,
        parse_params_func= parse_parameters,
        DeepGraphPredictorClass= DeepGraphPredictor,
        output_dir= output_dir
    )

Using device: cpu
Loading model from best_model_across_5folds.pth
Loading param file=/Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/DPAC1_parameters/1.py
Initial time=0 => saved to /Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/predictions
  -> Visualization saved for timepoint 0.

--- Predicting from time 0 to 100 ---
  -> Visualization saved for timepoint 100.

--- Predicting from time 100 to 200 ---
  -> Visualization saved for timepoint 200.

--- Predicting from time 200 to 300 ---
  -> Visualization saved for timepoint 300.

--- Predicting from time 300 to 400 ---
  -> Visualization saved for timepoint 400.

--- Predicting from time 400 to 500 ---


KeyboardInterrupt: 